**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 30: API 서빙 + Streamlit 웹앱 구현

## 🎯 모델 서빙과 웹앱 구현 개요

파인튜닝한 모델을 **실제로 사용 가능한 서비스**로 만들기 위해서는
**API 서버**와 **사용자 인터페이스(UI)**가 필요합니다.

### 이번 세션에서 배울 내용

- 1️⃣ FastAPI로 REST API 서버 구현
- 2️⃣ OpenAI 호환 API 형식
- 3️⃣ Ollama API 활용
- 4️⃣ Streamlit 기본 (st.chat_input, st.chat_message)
- 5️⃣ Streamlit 챗봇 구현 (대화 히스토리 관리)
- 6️⃣ 파인튜닝된 모델 연동
- 7️⃣ 배포 체크리스트

### 실습 환경
- **GPU**: 선택적 (Ollama 사용 시 필요, API만 구현 시 불필요)
- **핵심 패키지**: fastapi, uvicorn, streamlit, requests, openai

### 아키텍처 개요
```
[사용자] → [Streamlit UI] → [FastAPI/Ollama API] → [LLM 모델]
```

⚠️ **참고**: Streamlit과 FastAPI 서버는 노트북 안에서 직접 실행할 수 없습니다.
코드를 `.py` 파일로 저장한 후 터미널에서 실행해야 합니다.

In [1]:
# 🔧 필수 패키지 설치 (필요 시 주석 해제)
# !pip install fastapi uvicorn streamlit requests openai pydantic

print("✅ 패키지 설치 완료!")
print("\n📦 이번 세션에서 사용하는 패키지:")
print("  - fastapi: REST API 프레임워크")
print("  - uvicorn: ASGI 서버")
print("  - streamlit: 웹앱 프레임워크")
print("  - requests: HTTP 클라이언트")
print("  - openai: OpenAI API 클라이언트")
print("  - pydantic: 데이터 검증")

✅ 패키지 설치 완료!

📦 이번 세션에서 사용하는 패키지:
  - fastapi: REST API 프레임워크
  - uvicorn: ASGI 서버
  - streamlit: 웹앱 프레임워크
  - requests: HTTP 클라이언트
  - openai: OpenAI API 클라이언트
  - pydantic: 데이터 검증


In [2]:
# 📦 기본 라이브러리 임포트
import os
import json
import time

print("✅ 기본 라이브러리 임포트 완료!")

# GPU 모니터링 (선택적)
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print(f"[{tag}] GPU를 사용할 수 없습니다.")

print_gpu_memory("초기 상태")

✅ 기본 라이브러리 임포트 완료!
[초기 상태] GPU: 0.0GB / 23.5GB


---

## 1️⃣ FastAPI로 REST API 서버 구현

**FastAPI**는 Python에서 가장 인기 있는 비동기 웹 프레임워크로,
LLM 모델을 API로 서빙하기에 적합합니다.

### FastAPI 특징
- 🔹 **비동기(async)** 지원으로 높은 동시 처리 성능
- 🔹 **자동 API 문서** 생성 (Swagger UI)
- 🔹 **Pydantic** 기반 데이터 검증
- 🔹 **타입 힌트** 기반 직관적인 코드
- 🔹 **uvicorn** ASGI 서버로 실행

### 서버 구조
```
POST /chat    → 채팅 API (메시지 입력 → 응답 반환)
POST /generate → 텍스트 생성 API
GET  /health   → 서버 상태 확인
```

In [7]:
# 📝 FastAPI 서버 코드 작성
# 이 코드를 server.py로 저장하여 실행합니다.

fastapi_server_code = '''
# server.py - FastAPI LLM 서버
# 실행: uvicorn server:app --host 0.0.0.0 --port 9200

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time
import uvicorn

# ============================================
# 1. FastAPI 앱 생성
# ============================================
app = FastAPI(
    title="LLM API Server",
    description="파인튜닝된 LLM 모델 API 서버",
    version="1.0.0"
)

# ============================================
# 2. 요청/응답 스키마 정의
# ============================================
class Message(BaseModel):
    role: str          # "system", "user", "assistant"
    content: str

class ChatRequest(BaseModel):
    messages: List[Message]
    max_new_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9

class ChatResponse(BaseModel):
    response: str
    tokens_generated: int
    time_seconds: float

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: Optional[int] = 256
    temperature: Optional[float] = 0.7

# ============================================
# 3. 모델 로딩 (서버 시작 시 1회)
# ============================================
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

@app.on_event("startup")
async def load_model():
    global model, tokenizer
    print(f"모델 로딩 중: {MODEL_NAME}")
    
    # 4bit 양자화로 로딩 (RTX 4060 호환)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
    print("모델 로딩 완료!")

# ============================================
# 4. API 엔드포인트
# ============================================

@app.get("/health")
async def health_check():
    """서버 상태 확인"""
    return {
        "status": "healthy",
        "model": MODEL_NAME,
        "gpu_available": torch.cuda.is_available()
    }

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """채팅 API"""
    try:
        # 메시지를 chat template으로 변환
        messages = [{"role": m.role, "content": m.content} for m in request.messages]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        start_time = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=request.max_new_tokens,
                temperature=request.temperature,
                top_p=request.top_p,
                do_sample=request.temperature > 0,
            )
        elapsed = time.time() - start_time
        
        # 생성된 텍스트 추출
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        return ChatResponse(
            response=response_text,
            tokens_generated=len(new_tokens),
            time_seconds=round(elapsed, 2)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/generate")
async def generate(request: GenerateRequest):
    """텍스트 생성 API"""
    try:
        inputs = tokenizer(request.prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=request.max_new_tokens,
                temperature=request.temperature,
                do_sample=request.temperature > 0,
            )
        
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return {"generated_text": generated}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ============================================
# 5. 서버 실행
# ============================================
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=9200)
'''

# 파일로 저장
with open("server.py", "w", encoding="utf-8") as f:
    f.write(fastapi_server_code)

print("✅ server.py 파일 저장 완료!")
print("\n🚀 실행 방법:")
print("   터미널에서: uvicorn server:app --host 0.0.0.0 --port 9200")
print("   또는: python server.py")
print("\n📖 API 문서 확인: http://localhost:9200/docs")

✅ server.py 파일 저장 완료!

🚀 실행 방법:
   터미널에서: uvicorn server:app --host 0.0.0.0 --port 9200
   또는: python server.py

📖 API 문서 확인: http://localhost:9200/docs


In [ ]:
# 🧪 FastAPI 서버 테스트 코드 (서버가 실행 중일 때 사용)
import requests

API_URL = "http://localhost:9200"

def test_health():
    """서버 상태 확인"""
    try:
        response = requests.get(f"{API_URL}/health")
        print(f"✅ 서버 상태: {response.json()}")
        return True
    except requests.exceptions.ConnectionError:
        print("❌ 서버에 연결할 수 없습니다. 서버를 먼저 실행해주세요.")
        return False

def test_chat(message):
    """채팅 API 테스트"""
    payload = {
        "messages": [
            {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
            {"role": "user", "content": message}
        ],
        "max_new_tokens": 200,
        "temperature": 0.7
    }
    
    response = requests.post(f"{API_URL}/chat", json=payload)
    result = response.json()
    
    print(f"📝 질문: {message}")
    print(f"🤖 응답: {result['response']}")
    print(f"📊 토큰 수: {result['tokens_generated']}, 시간: {result['time_seconds']}초")
    return result

# 서버 테스트
print("🧪 FastAPI 서버 테스트")
print("=" * 60)
print("\n⚠️ 아래 테스트는 서버가 실행 중일 때만 동작합니다.")
print("   터미널에서 'uvicorn server:app --port 9200'을 먼저 실행하세요.")
print("\n---")

if test_health():
    test_chat("인공지능이 뭔가요?")
else:
    print("\n💡 서버 실행 후 이 셀을 다시 실행하세요.")

🧪 FastAPI 서버 테스트

⚠️ 아래 테스트는 서버가 실행 중일 때만 동작합니다.
   터미널에서 'uvicorn server:app --port 8000'을 먼저 실행하세요.

---
✅ 서버 상태: {'status': 'healthy', 'service': 'auth', 'version': '1.0.0'}
📝 질문: 인공지능이 뭔가요?


KeyError: 'response'

---

## 2️⃣ OpenAI 호환 API 형식

OpenAI API 형식은 **사실상의 표준**이 되었습니다.
우리 서버도 이 형식을 따르면 기존 OpenAI 클라이언트 코드를 **그대로 재사용**할 수 있습니다.

### OpenAI API 형식의 장점
- 🔹 **호환성**: 기존 OpenAI 클라이언트 라이브러리 사용 가능
- 🔹 **표준화**: 다양한 도구/프레임워크(LangChain 등)와 호환
- 🔹 **전환 용이**: OpenAI → 로컬 모델로 쉽게 전환 가능
- 🔹 **스트리밍**: SSE(Server-Sent Events) 기반 스트리밍 지원

In [ ]:
# 📝 OpenAI 호환 API 서버 코드

openai_compatible_code = '''
# openai_server.py - OpenAI 호환 API 서버
# 실행: uvicorn openai_server:app --host 0.0.0.0 --port 9200

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import List, Optional
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from threading import Thread
import time
import json
import uuid
import uvicorn

app = FastAPI(title="OpenAI Compatible API")

# ============================================
# OpenAI 호환 스키마
# ============================================
class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str = "local-model"
    messages: List[ChatMessage]
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9
    stream: Optional[bool] = False

# 모델 로딩
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

@app.on_event("startup")
async def startup():
    global model, tokenizer
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
    print("모델 로딩 완료!")

# ============================================
# OpenAI 호환 엔드포인트
# ============================================
@app.post("/v1/chat/completions")
async def chat_completions(request: ChatCompletionRequest):
    messages = [{"role": m.role, "content": m.content} for m in request.messages]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    if request.stream:
        # 스트리밍 응답
        return StreamingResponse(
            stream_generate(inputs, request),
            media_type="text/event-stream"
        )
    else:
        # 일반 응답
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=request.max_tokens,
                temperature=request.temperature,
                top_p=request.top_p,
                do_sample=request.temperature > 0,
            )
        
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        # OpenAI 형식으로 응답
        return {
            "id": f"chatcmpl-{uuid.uuid4().hex[:8]}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": request.model,
            "choices": [{
                "index": 0,
                "message": {
                    "role": "assistant",
                    "content": response_text
                },
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": inputs["input_ids"].shape[1],
                "completion_tokens": len(new_tokens),
                "total_tokens": inputs["input_ids"].shape[1] + len(new_tokens)
            }
        }

async def stream_generate(inputs, request):
    """스트리밍 생성"""
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    generation_kwargs = dict(
        **inputs,
        max_new_tokens=request.max_tokens,
        temperature=request.temperature,
        do_sample=request.temperature > 0,
        streamer=streamer,
    )
    
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    for text in streamer:
        chunk = {
            "id": f"chatcmpl-{uuid.uuid4().hex[:8]}",
            "object": "chat.completion.chunk",
            "created": int(time.time()),
            "model": request.model,
            "choices": [{
                "index": 0,
                "delta": {"content": text},
                "finish_reason": None
            }]
        }
        yield f"data: {json.dumps(chunk)}\\n\\n"
    
    yield "data: [DONE]\\n\\n"

# 모델 목록 (OpenAI 호환)
@app.get("/v1/models")
async def list_models():
    return {
        "object": "list",
        "data": [{
            "id": "local-model",
            "object": "model",
            "owned_by": "local"
        }]
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=9200)
'''

# 파일로 저장
with open("openai_server.py", "w", encoding="utf-8") as f:
    f.write(openai_compatible_code)

print("✅ openai_server.py 파일 저장 완료!")
print("\n🚀 실행 방법: uvicorn openai_server:app --port 9200")
print("📖 API 문서: http://localhost:9200/docs")

In [ ]:
# 🧪 OpenAI 호환 API 클라이언트 테스트
# ⚠️ openai_server.py가 포트 9200에서 실행 중이어야 합니다.
# 터미널에서: uvicorn openai_server:app --port 9200

from openai import OpenAI

# 로컬 서버를 가리키도록 설정
client = OpenAI(
    base_url="http://localhost:9200/v1",
    api_key="not-needed"
)

# 1. 일반 요청
print("📝 일반 요청 테스트")
print("="*60)
response = client.chat.completions.create(
    model="local-model",
    messages=[
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
        {"role": "user", "content": "인공지능의 미래에 대해 간단히 설명해주세요."}
    ],
    max_tokens=200,
    temperature=0.7
)
print(f"🤖 응답: {response.choices[0].message.content}")
print(f"📊 토큰: {response.usage.total_tokens}")

# 2. 스트리밍 요청
print("\n📝 스트리밍 요청 테스트")
print("="*60)
print("🤖 응답: ", end="")
stream = client.chat.completions.create(
    model="local-model",
    messages=[
        {"role": "user", "content": "파이썬이 뭔가요?"}
    ],
    stream=True
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

print("\n💡 핵심: base_url만 바꾸면 OpenAI → 로컬 모델로 전환 가능!")


---

## 3️⃣ Ollama API 활용

**Ollama**는 로컬 LLM 서빙을 가장 쉽게 할 수 있는 도구입니다.
설치 후 한 줄로 모델을 실행할 수 있으며, **OpenAI 호환 API**를 자동으로 제공합니다.

### Ollama 장점
- 🔹 **설치/사용이 매우 간편** (`ollama run qwen2.5:3b`)
- 🔹 **GGUF 기반** 양자화 모델 자동 관리
- 🔹 **OpenAI 호환 API** 자동 제공 (포트 11434)
- 🔹 **GPU 자동 감지** 및 메모리 관리
- 🔹 **Modelfile**로 커스텀 모델 생성 가능

In [ ]:
# 🔧 Ollama 기본 사용법
import subprocess

# Ollama 설치 확인
result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ Ollama 설치됨: {result.stdout.strip()}")
else:
    print("❌ Ollama가 설치되어 있지 않습니다.")
    print("   설치: curl -fsSL https://ollama.ai/install.sh | sh")

# 사용 가능한 모델 목록
print("\n📋 로컬 모델 목록:")
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout if result.stdout else "  (모델 없음 - ollama pull qwen2.5:1.5b 로 다운로드)")

# 모델 다운로드 (없으면)
print("\n⏳ qwen2.5:1.5b 모델 확인 중...")
result = subprocess.run(["ollama", "show", "qwen2.5:1.5b"], capture_output=True, text=True)
if result.returncode != 0:
    print("   모델이 없습니다. 다운로드하려면:")
    print("   ollama pull qwen2.5:1.5b")
else:
    print("   ✅ 모델 존재")


In [ ]:
# 🐍 Python에서 Ollama API 사용
import requests

# Ollama REST API로 요청
print("📝 Ollama REST API 테스트")
print("="*60)

try:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:1.5b",
            "prompt": "인공지능이란 무엇인가요?",
            "stream": False,
        },
        timeout=60
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"🤖 응답: {result['response'][:300]}")
        print(f"📊 생성 시간: {result.get('total_duration', 0) / 1e9:.1f}초")
    else:
        print(f"❌ 에러: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("⚠️ Ollama 서버가 실행 중이 아닙니다.")
    print("   터미널에서: ollama serve")


In [ ]:
# 🔗 Ollama의 OpenAI 호환 API 사용
from openai import OpenAI

# Ollama는 OpenAI 호환 API를 기본 제공 (포트 11434)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print("📝 Ollama OpenAI 호환 API 테스트")
print("="*60)

try:
    response = client.chat.completions.create(
        model="qwen2.5:1.5b",
        messages=[
            {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
            {"role": "user", "content": "LoRA 파인튜닝의 장점을 3가지 알려주세요."}
        ],
        max_tokens=300,
        temperature=0.7
    )
    print(f"🤖 응답: {response.choices[0].message.content}")
    
    print("\n💡 핵심: Ollama도 OpenAI 호환 API를 제공하므로 동일한 코드로 사용 가능!")
    print("   - FastAPI 서버: base_url='http://localhost:9200/v1'")
    print("   - Ollama: base_url='http://localhost:11434/v1'")
    print("   - OpenAI: base_url='https://api.openai.com/v1'")
except Exception as e:
    print(f"⚠️ Ollama 연결 실패: {e}")
    print("   ollama serve 실행 후 다시 시도하세요.")


---

## 4️⃣ Streamlit 기본 (st.chat_input, st.chat_message)

**Streamlit**은 Python만으로 웹앱을 만들 수 있는 프레임워크입니다.
LLM 챗봇 UI를 구축하기에 매우 적합합니다.

### Streamlit 채팅 관련 주요 컴포넌트
- 🔹 `st.chat_input()` - 사용자 입력 필드
- 🔹 `st.chat_message("user")` - 사용자 메시지 표시
- 🔹 `st.chat_message("assistant")` - AI 응답 표시
- 🔹 `st.session_state` - 대화 히스토리 관리
- 🔹 `st.write_stream()` - 스트리밍 응답 표시

⚠️ **중요**: Streamlit 앱은 `.py` 파일로 저장 후 `streamlit run app.py`로 실행해야 합니다.

In [ ]:
# 📝 Streamlit 기본 채팅 앱 코드

streamlit_basic_code = '''
# chat_basic.py - Streamlit 기본 채팅 앱
# 실행: streamlit run chat_basic.py

import streamlit as st

# ============================================
# 1. 페이지 설정
# ============================================
st.set_page_config(
    page_title="LLM 챗봇",
    page_icon="🤖",
    layout="centered"
)

st.title("🤖 LLM 챗봇")
st.caption("파인튜닝된 모델과 대화해보세요!")

# ============================================
# 2. 세션 상태 초기화 (대화 히스토리)
# ============================================
if "messages" not in st.session_state:
    st.session_state.messages = []

# ============================================
# 3. 기존 대화 히스토리 표시
# ============================================
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# ============================================
# 4. 사용자 입력 처리
# ============================================
if prompt := st.chat_input("메시지를 입력하세요..."):
    # 사용자 메시지 표시
    with st.chat_message("user"):
        st.markdown(prompt)
    
    # 히스토리에 추가
    st.session_state.messages.append({"role": "user", "content": prompt})
    
    # AI 응답 생성 (여기서는 에코 - 나중에 실제 모델 연동)
    response = f"에코: {prompt}"  # 임시 응답
    
    # AI 응답 표시
    with st.chat_message("assistant"):
        st.markdown(response)
    
    # 히스토리에 추가
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

# 파일로 저장
with open("chat_basic.py", "w", encoding="utf-8") as f:
    f.write(streamlit_basic_code)

print("✅ chat_basic.py 파일 저장 완료!")
print("\n🚀 실행 방법:")
print("   터미널에서: streamlit run chat_basic.py")
print("   브라우저에서: http://localhost:8501")
print("\n📝 이 앱은 에코(echo) 기능만 있습니다.")
print("   다음 단계에서 실제 LLM을 연동합니다.")

In [ ]:
# 📝 Streamlit 주요 컴포넌트 설명
print("📝 Streamlit 챗봇 주요 컴포넌트")
print("=" * 60)

components = {
    "st.chat_input()": "사용자 입력 필드 (하단 고정)",
    "st.chat_message('user')": "사용자 메시지 박스",
    "st.chat_message('assistant')": "AI 응답 메시지 박스",
    "st.session_state": "상태 관리 (대화 히스토리 저장)",
    "st.markdown()": "마크다운 렌더링",
    "st.write_stream()": "스트리밍 텍스트 표시",
    "st.sidebar": "사이드바 (설정 패널)",
    "st.spinner()": "로딩 표시",
    "st.set_page_config()": "페이지 설정 (제목, 아이콘, 레이아웃)",
}

for comp, desc in components.items():
    print(f"  🔹 {comp:35s} → {desc}")

print("\n💡 Streamlit은 Python 코드 변경 시 자동으로 리로드됩니다!")
print("💡 'streamlit run app.py --server.runOnSave true'로 자동 저장 리로드")

---

## 5️⃣ Streamlit 챗봇 구현 (대화 히스토리 관리)

실제 LLM API와 연동하는 완전한 챗봇을 구현합니다.

### 구현 기능
- 🔹 대화 히스토리 관리
- 🔹 시스템 프롬프트 설정
- 🔹 모델/파라미터 조절 (사이드바)
- 🔹 스트리밍 응답
- 🔹 대화 초기화

In [ ]:
# 📝 완전한 Streamlit 챗봇 (Ollama 연동)

streamlit_chatbot_code = '''
# chatbot.py - Streamlit LLM 챗봇 (Ollama 연동)
# 실행: streamlit run chatbot.py
# 사전 요구: ollama serve && ollama pull qwen2.5:1.5b

import streamlit as st
from openai import OpenAI

# ============================================
# 1. 페이지 설정
# ============================================
st.set_page_config(
    page_title="AI 챗봇",
    page_icon="🤖",
    layout="centered"
)

# ============================================
# 2. 사이드바 - 설정
# ============================================
with st.sidebar:
    st.header("⚙️ 설정")
    
    # API 백엔드 선택
    backend = st.selectbox(
        "API 백엔드",
        ["Ollama (로컬)", "FastAPI (커스텀)", "OpenAI"]
    )
    
    if backend == "Ollama (로컬)":
        base_url = "http://localhost:11434/v1"
        api_key = "ollama"
        model = st.selectbox("모델", ["qwen2.5:1.5b", "qwen2.5:3b", "qwen2.5:7b"])
    elif backend == "FastAPI (커스텀)":
        base_url = st.text_input("API URL", "http://localhost:9200/v1")
        api_key = "not-needed"
        model = "local-model"
    else:
        base_url = "https://api.openai.com/v1"
        api_key = st.text_input("API Key", type="password")
        model = st.selectbox("모델", ["gpt-4o-mini", "gpt-4o", "gpt-3.5-turbo"])
    
    st.divider()
    
    # 모델 파라미터
    temperature = st.slider("Temperature", 0.0, 2.0, 0.7, 0.1)
    max_tokens = st.slider("Max Tokens", 50, 2048, 512, 50)
    
    # 시스템 프롬프트
    system_prompt = st.text_area(
        "시스템 프롬프트",
        value="당신은 유용하고 친절한 AI 어시스턴트입니다. 한국어로 답변합니다.",
        height=100
    )
    
    st.divider()
    
    # 대화 초기화 버튼
    if st.button("🗑️ 대화 초기화", use_container_width=True):
        st.session_state.messages = []
        st.rerun()

# ============================================
# 3. 메인 화면
# ============================================
st.title("🤖 AI 챗봇")
st.caption(f"백엔드: {backend} | 모델: {model}")

# ============================================
# 4. OpenAI 클라이언트 생성
# ============================================
client = OpenAI(base_url=base_url, api_key=api_key)

# ============================================
# 5. 세션 상태 초기화
# ============================================
if "messages" not in st.session_state:
    st.session_state.messages = []

# ============================================
# 6. 대화 히스토리 표시
# ============================================
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# ============================================
# 7. 사용자 입력 처리
# ============================================
if prompt := st.chat_input("메시지를 입력하세요..."):
    # 사용자 메시지 표시 및 저장
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)
    
    # API 요청용 메시지 구성 (시스템 프롬프트 포함)
    api_messages = [{"role": "system", "content": system_prompt}]
    api_messages.extend(st.session_state.messages)
    
    # AI 응답 생성 (스트리밍)
    with st.chat_message("assistant"):
        try:
            stream = client.chat.completions.create(
                model=model,
                messages=api_messages,
                temperature=temperature,
                max_tokens=max_tokens,
                stream=True
            )
            response = st.write_stream(
                chunk.choices[0].delta.content
                for chunk in stream
                if chunk.choices[0].delta.content is not None
            )
        except Exception as e:
            response = f"❌ 오류: {str(e)}"
            st.error(response)
    
    # 응답 저장
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

# 파일로 저장
with open("chatbot.py", "w", encoding="utf-8") as f:
    f.write(streamlit_chatbot_code)

print("✅ chatbot.py 파일 저장 완료!")
print("\n🚀 실행 방법:")
print("   1. 터미널 1: ollama serve")
print("   2. 터미널 2: streamlit run chatbot.py")
print("   3. 브라우저: http://localhost:8501")
print("\n✨ 주요 기능:")
print("   - 사이드바에서 백엔드/모델/파라미터 변경")
print("   - 스트리밍 응답")
print("   - 대화 히스토리 관리")
print("   - 대화 초기화")

---

## 6️⃣ 파인튜닝된 모델 연동

이전 세션에서 파인튜닝한 모델을 서빙하는 방법을 알아봅니다.

### 연동 방식

| 방법 | 적합한 경우 |
|------|----------|
| **Transformers + FastAPI** | 직접 제어가 필요할 때 |
| **GGUF + Ollama** | 간편한 서빙이 필요할 때 |
| **vLLM** | 프로덕션 서빙 (높은 처리량) |

### 파인튜닝 모델 → 서빙 파이프라인
```
[LoRA 어댑터] → [모델 병합] → [양자화 (GGUF/GPTQ)] → [서빙 (Ollama/vLLM/FastAPI)]
```

In [ ]:
# 📝 파인튜닝 모델 병합 및 서빙 파이프라인
print("="*60)
print("📌 파인튜닝 모델 → 서빙 파이프라인")
print("="*60)

import os
from pathlib import Path

# LoRA 어댑터 확인
adapter_dirs = list(Path("./output").glob("**/lora_adapter")) + list(Path("./output").glob("**/checkpoint*"))
print(f"\n📋 발견된 어댑터/체크포인트: {len(adapter_dirs)}개")
for d in adapter_dirs:
    size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1024 / 1024
    print(f"   📁 {d} ({size:.1f}MB)")

if not adapter_dirs:
    print("   ⚠️ 어댑터 없음 — 이전 노트북에서 파인튜닝 실행 필요")

print("\n📌 서빙 방법:")
print("   1. LoRA 어댑터 + base 모델을 merge → 단일 모델로 저장")
print("   2. merged 모델을 FastAPI 서버에 로드")
print("   3. 또는 GGUF로 변환 → Ollama에 등록")

print("\n📌 merge 코드 예시:")
print("""
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = PeftModel.from_pretrained(base_model, "./output/.../lora_adapter")
merged = model.merge_and_unload()
merged.save_pretrained("./output/merged_model")
""")


In [ ]:
# 📝 파인튜닝 모델을 직접 로딩하는 FastAPI 서버

finetuned_server_code = '''
# finetuned_server.py - 파인튜닝 모델 FastAPI 서버
# 실행: uvicorn finetuned_server:app --port 8002

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import time

app = FastAPI(title="Fine-tuned LLM API")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[ChatMessage]
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.7

# 설정
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_ADAPTER = "./my-lora-adapter"  # LoRA 어댑터 경로
# 또는 병합된 모델 경로
# MERGED_MODEL = "./merged-model"

@app.on_event("startup")
async def load_model():
    global model, tokenizer
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    
    # 방법 1: LoRA 어댑터 로딩
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto"
    )
    model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
    
    # 방법 2: 병합된 모델 직접 로딩 (더 빠름)
    # model = AutoModelForCausalLM.from_pretrained(
    #     MERGED_MODEL, quantization_config=bnb_config, device_map="auto"
    # )
    
    print("파인튜닝 모델 로딩 완료!")

@app.post("/v1/chat/completions")
async def chat(request: ChatRequest):
    messages = [{"role": m.role, "content": m.content} for m in request.messages]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_tokens,
            temperature=request.temperature,
            do_sample=request.temperature > 0,
        )
    
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    return {
        "choices": [{"message": {"role": "assistant", "content": response_text}}]
    }
'''

# 파일로 저장
with open("finetuned_server.py", "w", encoding="utf-8") as f:
    f.write(finetuned_server_code)

print("✅ finetuned_server.py 파일 저장 완료!")
print("\n🚀 실행 방법: uvicorn finetuned_server:app --port 8002")
print("\n💡 LORA_ADAPTER 경로를 실제 어댑터 경로로 변경하세요.")
print("💡 Streamlit 챗봇에서 FastAPI 백엔드 URL을 8002번으로 설정하면 연동됩니다.")

---

## 7️⃣ 배포 체크리스트

모델을 실제 서비스로 배포하기 전에 확인해야 할 사항들을 정리합니다.

### 배포 전 체크리스트

#### 🔹 모델 준비
- [ ] LoRA 어댑터가 기본 모델에 병합되었는가?
- [ ] 양자화가 적용되었는가? (GPTQ/AWQ/GGUF)
- [ ] 추론 품질 테스트를 완료했는가?
- [ ] 모델 크기가 서버 VRAM에 적합한가?

#### 🔹 API 서버
- [ ] 에러 핸들링이 구현되어 있는가?
- [ ] 입력 길이 제한이 설정되어 있는가?
- [ ] Rate limiting이 적용되어 있는가?
- [ ] 로깅/모니터링이 설정되어 있는가?
- [ ] Health check 엔드포인트가 있는가?

#### 🔹 보안
- [ ] API 키/인증이 적용되어 있는가?
- [ ] CORS 설정이 적절한가?
- [ ] 입력 검증(sanitization)이 구현되어 있는가?
- [ ] 프롬프트 인젝션 방어가 있는가?

#### 🔹 성능
- [ ] 동시 요청 처리가 가능한가?
- [ ] 응답 시간이 허용 범위 내인가?
- [ ] GPU 메모리 누수가 없는가?
- [ ] 스트리밍이 지원되는가?

## 📋 배포 환경별 권장 구성

| 환경 | 서빙 방법 | 장점 | 단점 |
|------|----------|------|------|
| **개발/테스트** | FastAPI + transformers | 간단, 빠른 설정 | 느림, 단일 요청 |
| **소규모 서비스** | Ollama | 설치 쉬움, GGUF 지원 | 커스텀 제한 |
| **프로덕션** | vLLM | 높은 처리량, PagedAttention | 설정 복잡 |
| **웹 데모** | Streamlit + API | 빠른 프로토타이핑 | 동시 접속 제한 |

### 권장 워크플로우
```
파인튜닝 → LoRA merge → GGUF 변환 → Ollama 등록 → Streamlit 연동
```


### 🐳 Docker 배포 예시

```dockerfile
# Dockerfile
FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY server.py .
COPY model/ ./model/

EXPOSE 9200
CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "9200"]
```

```bash
# 빌드 및 실행
docker build -t llm-server .
docker run --gpus all -p 9200:9200 llm-server
```


---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **FastAPI 서버**: REST API로 LLM 서빙, Pydantic 스키마 활용

2️⃣ **OpenAI 호환 API**: `/v1/chat/completions` 형식으로 기존 코드 재사용

3️⃣ **Ollama**: 가장 간편한 로컬 서빙, OpenAI 호환 API 자동 제공

4️⃣ **Streamlit 기본**: `st.chat_input()`, `st.chat_message()`, `st.session_state`

5️⃣ **Streamlit 챗봇**: 대화 히스토리, 스트리밍, 사이드바 설정

6️⃣ **파인튜닝 모델 연동**: LoRA 병합 → 양자화 → 서빙

7️⃣ **배포 체크리스트**: 보안, 성능, 모니터링 확인

### 생성된 파일 목록
- `server.py` - FastAPI 기본 서버
- `openai_server.py` - OpenAI 호환 서버 (스트리밍 포함)
- `chat_basic.py` - Streamlit 기본 채팅 앱
- `chatbot.py` - Streamlit 완전한 챗봇 (Ollama 연동)
- `finetuned_server.py` - 파인튜닝 모델 서버

### 다음 세션 예고
- 🔜 **Session 31**: 평가 메트릭, LLM-as-a-Judge

In [ ]:
# 📋 생성된 파일 확인
import os

files_created = [
    "server.py",
    "openai_server.py",
    "chat_basic.py",
    "chatbot.py",
    "finetuned_server.py"
]

print("📋 생성된 파일 목록")
print("=" * 60)
for f in files_created:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    size = os.path.getsize(f) if exists else 0
    print(f"  {status} {f:30s} ({size:,} bytes)")

print("\n" + "=" * 60)
print("✅ Session 30: API 서빙 + Streamlit 웹앱 구현 완료!")
print("=" * 60)
print("\n📌 다음 세션: 평가 메트릭, LLM-as-a-Judge")
print("📌 준비물: OpenAI API 키, matplotlib")